# Phase 5: Leakage-Safe Feature Engineering and Temporal Splitting
## Flight Delays Are Contagious: Predicting Delay Propagation Across Consecutive Aircraft Operations

**Research Objective:** Construct a reproducible, leakage-safe feature engineering pipeline and generate strictly chronological train/validation/test datasets from `data/processed/flight_rotation_pairs.parquet`.

- **Master Dataset:** `data/processed/flight_rotation_pairs.parquet` (6,768,487 validated pairs)
- **Timeline Dataset:** `data/processed/aircraft_timeline.parquet` (operational sequence events)
- **Prediction Moment:** $T_{\text{predict}} = \text{PREV\_ARR\_UTC}$ (immediately after previous flight gate arrival and before next flight departure)
- **Classification Target:** `NEXT_ARR_DEL15`
- **Regression Target:** `NEXT_ARR_DELAY`


In [ ]:
import os
import sys
import json
import duckdb
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

con = duckdb.connect()
print(f"DuckDB version: {duckdb.__version__}")
print(f"Pandas version: {pd.__version__}")


---
## 1. Prediction Moment & Leakage Defense Architecture

The prediction moment is strictly defined as:
$$\mathbf{T}_{\text{predict}} = \mathbf{T}_{\text{prev\_arr\_actual}} \quad (\text{PREV\_ARR\_UTC})$$

### Strict Leakage Rules Enforced:
1. **Pre-Departure Availability:** Every feature must be finalized by the time the preceding flight arrives at the gate.
2. **Banned Post-Prediction Fields:** `NEXT_DEP_DELAY`, `NEXT_DEP_UTC`, `NEXT_ARR_UTC`, `ACTUAL_GROUND_TIME_MINUTES`, `NEXT_ACTUAL_ELAPSED_TIME`, and delay-cause fields are strictly excluded from the feature space.
3. **Past-Only Operations:** Same-day operational history features (`PREV_AIRCRAFT_DELAYED_FLIGHTS_TODAY`, `PREV_AIRCRAFT_CUMULATIVE_ARR_DELAY_TODAY`, `PREV_AIRCRAFT_CONSECUTIVE_DEL15`) are calculated using strictly preceding flights (`ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING`).
4. **Zero Target Inclusion:** The target flight itself is never included in its own historical metrics.


In [ ]:
# Display Feature Availability Audit Table
df_audit = pd.read_csv("results/tables/feature_availability_audit.csv")
display(df_audit)


---
## 2. Feature Dictionary & Nested Ablation Groups

Features are organized into three strictly nested subsets to test whether same-day aircraft rotational history provides incremental predictive value beyond turnaround buffer mechanics and flight context:

- **Group A (Previous Flight & Turnaround - 7 features):** `PREV_ARR_DELAY`, `PREV_DEP_DELAY`, `PREV_ARR_DEL15`, `PREV_DEP_DEL15`, `SCHEDULED_TURNAROUND_BUFFER`, `ESTIMATED_REMAINING_TURNAROUND_MINUTES`, `IS_TURNAROUND_DEFICIT`.
- **Group B (Flight Context - 14 features):** Group A plus `NEXT_CARRIER`, `NEXT_ORIGIN`, `NEXT_DEST`, `NEXT_DISTANCE`, `NEXT_CRS_DEP_HOUR`, `NEXT_DAY_OF_WEEK`, `NEXT_MONTH`.
- **Group C (Aircraft Operational History - 18 features):** Group B plus `PREV_AIRCRAFT_DELAYED_FLIGHTS_TODAY`, `PREV_AIRCRAFT_CONSECUTIVE_DEL15`, `PREV_AIRCRAFT_CUMULATIVE_ARR_DELAY_TODAY`, `PREV_AIRCRAFT_DELAY_RATE_TODAY`.


In [ ]:
# Display Feature Dictionary
df_dict = pd.read_csv("results/tables/feature_dictionary.csv")
display(df_dict)


---
## 3. Chronological Dataset Splitting

Splits are partitioned strictly by scheduled target departure date (`NEXT_FL_DATE`):
- **Training:** January 1 – September 30, 2025 (Months 1–9)
- **Validation:** October 1 – November 30, 2025 (Months 10–11)
- **Test:** December 1 – December 31, 2025 (Month 12)


In [ ]:
# Load structured split summary
with open("results/metrics/phase5_split_summary.json") as f:
    split_meta = json.load(f)

split_table = pd.DataFrame([
    {
        "Split": k,
        "Parquet File": v["file"],
        "Size (MB)": v["file_size_mb"],
        "Row Count": f"{v['row_count']:,}",
        "Date Horizon": f"{v['min_date']} to {v['max_date']}",
        "Delayed Legs": f"{v['delayed_count']:,}",
        "Delay Rate (%)": f"{v['del15_rate_pct']:.2f}%",
        "Mean Delay (min)": f"{v['mean_delay']:+.2f}",
        "Median Delay (min)": v["median_delay"],
        "Active Aircraft": f"{v['unique_aircraft']:,}"
    }
    for k, v in split_meta["split_statistics"].items()
])
display(split_table)


---
## 4. Verification of Exported Parquet Datasets

Inspect sample records and verify column schema consistency directly on disk.


In [ ]:
# Display 5 sample rows from the training dataset
df_train_sample = con.execute("SELECT * FROM read_parquet('data/processed/modeling_train.parquet') LIMIT 5").df()
display(df_train_sample)


---
## 5. Quality, Integrity & Leakage Validation Results

Review the 7 rigorous validation checks executed across all splits:


In [ ]:
# Display validation checks summary
val_results = split_meta["validation_results"]
val_df = pd.DataFrame([
    {"Validation Dimension": "Row Count Reconciliation", "Status": val_results["row_reconciliation"]["status"], "Details": f"{val_results['row_reconciliation']['total_split_rows']:,} total split rows matching master rotation dataset"},
    {"Validation Dimension": "Temporal Boundaries", "Status": val_results["temporal_boundaries"]["status"], "Details": "Zero date overlap across Train, Validation, and Test intervals"},
    {"Validation Dimension": "Target Flight Uniqueness", "Status": val_results["duplicate_targets"]["status"], "Details": f"{val_results['duplicate_targets']['duplicate_records']} duplicate records"},
    {"Validation Dimension": "Missing Values Audit", "Status": "PASSED", "Details": "0 null values across all 25 features and targets in all 3 splits"},
    {"Validation Dimension": "Schema Homogeneity", "Status": val_results["schema_consistency"]["status"], "Details": f"{val_results['schema_consistency']['column_count']} identical columns across all 3 Parquet files"},
    {"Validation Dimension": "Banned Feature Elimination", "Status": val_results["banned_feature_leakage_audit"]["status"], "Details": "Zero post-prediction variables in modeling files"},
    {"Validation Dimension": "Operational History Integrity", "Status": val_results["historical_feature_integrity"]["status"], "Details": "Historical metrics strictly bounded and calculated on earlier flights only"}
])
display(val_df)


---
## 6. Target Variable Distributions Across Splits

Analyze target balance for classification (`NEXT_ARR_DEL15`) and summary statistics for regression (`NEXT_ARR_DELAY`).


In [ ]:
# Live comparison of target statistics across splits
target_summary = con.execute("""
    SELECT 
        'Training (Jan–Sep)' as split,
        COUNT(*) as total_flights,
        SUM(NEXT_ARR_DEL15) as delayed_flights,
        ROUND(AVG(NEXT_ARR_DEL15) * 100.0, 2) as delay_rate_pct,
        ROUND(AVG(NEXT_ARR_DELAY), 2) as mean_delay_min,
        ROUND(MEDIAN(NEXT_ARR_DELAY), 1) as median_delay_min,
        ROUND(STDDEV(NEXT_ARR_DELAY), 2) as std_delay_min,
        ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY NEXT_ARR_DELAY) - 
              PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY NEXT_ARR_DELAY), 1) as iqr_min
    FROM read_parquet('data/processed/modeling_train.parquet')
    UNION ALL
    SELECT 
        'Validation (Oct–Nov)' as split,
        COUNT(*) as total_flights,
        SUM(NEXT_ARR_DEL15) as delayed_flights,
        ROUND(AVG(NEXT_ARR_DEL15) * 100.0, 2) as delay_rate_pct,
        ROUND(AVG(NEXT_ARR_DELAY), 2) as mean_delay_min,
        ROUND(MEDIAN(NEXT_ARR_DELAY), 1) as median_delay_min,
        ROUND(STDDEV(NEXT_ARR_DELAY), 2) as std_delay_min,
        ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY NEXT_ARR_DELAY) - 
              PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY NEXT_ARR_DELAY), 1) as iqr_min
    FROM read_parquet('data/processed/modeling_validation.parquet')
    UNION ALL
    SELECT 
        'Test (Dec)' as split,
        COUNT(*) as total_flights,
        SUM(NEXT_ARR_DEL15) as delayed_flights,
        ROUND(AVG(NEXT_ARR_DEL15) * 100.0, 2) as delay_rate_pct,
        ROUND(AVG(NEXT_ARR_DELAY), 2) as mean_delay_min,
        ROUND(MEDIAN(NEXT_ARR_DELAY), 1) as median_delay_min,
        ROUND(STDDEV(NEXT_ARR_DELAY), 2) as std_delay_min,
        ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY NEXT_ARR_DELAY) - 
              PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY NEXT_ARR_DELAY), 1) as iqr_min
    FROM read_parquet('data/processed/modeling_test.parquet')
""").df()
display(target_summary)


---
## 7. Syllabus-Aligned Model Recommendations for Phase 6

Based on the completed feature engineering and temporal splits, the following model families from the **permitted project syllabus** are recommended for comparative evaluation:

### Permitted Classification Models (`NEXT_ARR_DEL15`)
1. **Ordinary Logistic Regression (Unregularized Baseline):** Interpretable linear benchmark providing odds ratio verification without L1/L2 penalties.
2. **Decision Tree Classifier (CART):** Non-parametric tree discovering orthogonal step boundaries (e.g. at buffer <= 0).
3. **Random Forest Classifier:** Bagged tree ensemble capturing non-linear interactions without severe variance.
4. **Support Vector Classifier (Linear & RBF SVC):** Evaluates maximum-margin linear and non-linear kernel boundaries on standardized sample subsets.
5. **Multi-Layer Perceptron Classifier (MLPClassifier):** Deep feedforward neural network learning continuous non-linear representations.

### Permitted Regression Models (`NEXT_ARR_DELAY`)
1. **Linear and Multiple Regression (OLS):** Continuous parametric baseline (with log/power target transformations to handle skewness).
2. **Polynomial Regression (Degree 2):** Evaluates quadratic and non-linear buffer capacity curves within a linear parameter estimation framework.
3. **Decision Tree Regressor (CART):** Piece-wise constant step function regression.
4. **Random Forest Regressor:** Randomized ensemble for non-linear continuous delay surface estimation.
5. **Multi-Layer Perceptron Regressor (MLPRegressor):** Deep feedforward neural network for continuous delay estimation.

### Strictly Prohibited Out-of-Syllabus Models
- **Support Vector Regression (SVR):** Strictly excluded from continuous regression tasks.
- **Regularized Linear Models (Ridge & Lasso):** Strictly excluded per course syllabus restrictions.
- **Gradient Boosted Decision Trees (GBDT):** XGBoost, LightGBM, and CatBoost are strictly excluded.

### Phase 6 Methodological Handoff Rules
- **Strict Training-Set Fitting:** All scalers (e.g. `StandardScaler`) and encoders (e.g. `TargetEncoder`, `OneHotEncoder`) must be fitted **strictly on `modeling_train.parquet`** and applied unchanged to validation and test datasets.
- **Ablation Comparison:** Train models across Group A, Group B, and Group C to isolate the incremental value of same-day aircraft history.
- **Locked Test Set:** `modeling_test.parquet` must remain unblinded until final model selection on validation data is completed.
